# Paper Trading Demo (M27-M28)

This notebook demonstrates the paper trading infrastructure:
- **M27**: Trading loop, order routing, signal generation
- **M28**: State persistence, crash recovery, position tracking, monitoring

## Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                        TradingLoop                              │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐    │
│  │ Surface  │ → │ Signal   │ → │ Order    │ → │ Order    │    │
│  │ Provider │   │Generator │   │Generator │   │ Router   │    │
│  └──────────┘   └──────────┘   └──────────┘   └──────────┘    │
│        ↓              ↓              ↓              ↓          │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐    │
│  │ Position │   │  State   │   │ Trading  │   │   Kill   │    │
│  │ Tracker  │   │ Manager  │   │ Monitor  │   │  Switch  │    │
│  └──────────┘   └──────────┘   └──────────┘   └──────────┘    │
└─────────────────────────────────────────────────────────────────┘
```

In [1]:
import sys
sys.path.insert(0, '..')

import tempfile
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Optional

import pandas as pd

from src.config.schema import (
    DriftBandsConfig,
    ExecutionConfig,
    ExitSignalsConfig,
    KillSwitchConfig,
    PaperConfig,
    PositionManagementConfig,
    RebalancingConfig,
    RiskCapsConfig,
    RiskConfig,
    StressConfig,
)
from src.live.loop import TradingLoop
from src.live.router import PaperOrderRouter, Fill
from src.live.state import StateManager, TradingState
from src.live.positions import PositionTracker, PositionSnapshot
from src.live.monitoring import TradingMonitor, TradingMetrics, AlertLevel
from src.risk.portfolio import Portfolio
from src.strategy.types import Greeks, OptionLeg, OptionRight, Signal, SignalType

print("Imports successful!")

Imports successful!


## 1. Mock Providers

Create mock surface and signal providers for demonstration.

In [2]:
class MockSurfaceProvider:
    """Provides mock volatility surface data."""
    
    def __init__(self) -> None:
        self._call_count = 0
    
    def get_latest_surface(self) -> pd.DataFrame:
        """Return mock surface with price evolution."""
        self._call_count += 1
        # Simulate price movement
        price_drift = self._call_count * 0.05
        
        return pd.DataFrame([
            {
                "option_symbol": "SPY240315C00450000",
                "tenor_days": 30,
                "delta_bucket": "ATM",
                "strike": 450.0,
                "expiry": date(2024, 3, 15),
                "right": "C",
                "bid": 5.00 + price_drift,
                "ask": 5.10 + price_drift,
                "delta": 0.50,
                "gamma": 0.02,
                "vega": 0.30,
                "theta": -0.05,
                "iv": 0.20,
                "underlying_price": 450.0 + self._call_count,
            },
            {
                "option_symbol": "SPY240315C00455000",
                "tenor_days": 30,
                "delta_bucket": "25C",
                "strike": 455.0,
                "expiry": date(2024, 3, 15),
                "right": "C",
                "bid": 3.00 + price_drift * 0.6,
                "ask": 3.08 + price_drift * 0.6,
                "delta": 0.35,
                "gamma": 0.018,
                "vega": 0.25,
                "theta": -0.04,
                "iv": 0.18,
                "underlying_price": 450.0 + self._call_count,
            },
            {
                "option_symbol": "SPY240315P00445000",
                "tenor_days": 30,
                "delta_bucket": "25P",
                "strike": 445.0,
                "expiry": date(2024, 3, 15),
                "right": "P",
                "bid": 2.80 - price_drift * 0.3,
                "ask": 2.88 - price_drift * 0.3,
                "delta": -0.30,
                "gamma": 0.016,
                "vega": 0.22,
                "theta": -0.03,
                "iv": 0.19,
                "underlying_price": 450.0 + self._call_count,
            },
        ])


class MockSignalGenerator:
    """Generates mock trading signals."""
    
    def __init__(self, max_signals: int = 2) -> None:
        self._call_count = 0
        self._max_signals = max_signals
    
    def generate_signals(
        self,
        surface: pd.DataFrame,
        features: Optional[pd.DataFrame] = None,
    ) -> list[Signal]:
        """Generate a signal on first few calls."""
        self._call_count += 1
        
        if self._call_count <= self._max_signals:
            return [
                Signal(
                    signal_type=SignalType.DIRECTIONAL_VOL,
                    edge=0.05,
                    confidence=0.8,
                    tenor_days=30,
                    delta_bucket="ATM",
                )
            ]
        return []


# Test the providers
surface_provider = MockSurfaceProvider()
signal_generator = MockSignalGenerator(max_signals=2)

surface = surface_provider.get_latest_surface()
print(f"Surface shape: {surface.shape}")
print(f"\nSurface columns: {list(surface.columns)}")
print(f"\nFirst row:")
surface.iloc[0]

Surface shape: (3, 14)

Surface columns: ['option_symbol', 'tenor_days', 'delta_bucket', 'strike', 'expiry', 'right', 'bid', 'ask', 'delta', 'gamma', 'vega', 'theta', 'iv', 'underlying_price']

First row:


option_symbol       SPY240315C00450000
tenor_days                          30
delta_bucket                       ATM
strike                           450.0
expiry                      2024-03-15
right                                C
bid                               5.05
ask                               5.15
delta                              0.5
gamma                             0.02
vega                               0.3
theta                            -0.05
iv                                 0.2
underlying_price                 451.0
Name: 0, dtype: object

## 2. Configuration

Set up paper trading and risk configurations.

In [3]:
# Create temporary state directory
temp_dir = tempfile.mkdtemp()
state_dir = Path(temp_dir) / "state"
state_dir.mkdir()

print(f"State directory: {state_dir}")

# Paper trading config
paper_config = PaperConfig(
    state_dir=str(state_dir),
    save_state_interval=1,  # Save after every iteration
    loop_interval_seconds=0,  # No delay between iterations
    max_loop_iterations=5,  # Run 5 iterations
    halt_on_error=True,
)

# Execution config
execution_config = ExecutionConfig(
    slippage_bps=5.0,
    fee_per_contract=0.65,
)

# Risk config
risk_config = RiskConfig(
    caps=RiskCapsConfig(
        max_abs_delta=1000.0,
        max_abs_gamma=100.0,
        max_abs_vega=500.0,
        max_daily_loss=5000.0,
    ),
    kill_switch=KillSwitchConfig(
        halt_on_daily_loss=True,
        max_daily_loss=5000.0,
        halt_on_stress_breach=False,
        halt_on_liquidity_collapse=True,
        max_spread_pct=0.5,
    ),
    stress=StressConfig(enabled=False),
    position_management=PositionManagementConfig(
        exit_signals=ExitSignalsConfig(),
        rebalancing=RebalancingConfig(enabled=False),
        drift_bands=DriftBandsConfig(),
    ),
)

print("\nPaper Config:")
print(f"  - State dir: {paper_config.state_dir}")
print(f"  - Save interval: {paper_config.save_state_interval}")
print(f"  - Max iterations: {paper_config.max_loop_iterations}")

print("\nRisk Caps:")
print(f"  - Max delta: {risk_config.caps.max_abs_delta}")
print(f"  - Max vega: {risk_config.caps.max_abs_vega}")
print(f"  - Max daily loss: {risk_config.caps.max_daily_loss}")

State directory: /var/folders/ky/gxh1dy7j5p98_qx7tvmy00780000gn/T/tmp463ew9bq/state

Paper Config:
  - State dir: /var/folders/ky/gxh1dy7j5p98_qx7tvmy00780000gn/T/tmp463ew9bq/state
  - Save interval: 1
  - Max iterations: 5

Risk Caps:
  - Max delta: 1000.0
  - Max vega: 500.0
  - Max daily loss: 5000.0


## 3. Trading Loop Demo

Run the paper trading loop for 5 iterations.

In [4]:
# Create fresh providers
surface_provider = MockSurfaceProvider()
signal_generator = MockSignalGenerator(max_signals=2)
order_router = PaperOrderRouter(execution_config)

# Create trading loop
loop = TradingLoop(
    paper_config=paper_config,
    execution_config=execution_config,
    risk_config=risk_config,
    surface_provider=surface_provider,
    signal_generator=signal_generator,
    order_router=order_router,
)

print("Trading loop created")
print(f"Initial state: iteration={loop.state.iteration}, is_running={loop.state.is_running}")

Trading loop created
Initial state: iteration=0, is_running=False


In [5]:
# Run the trading loop
print("Starting trading loop...\n")
loop.start()
print("\nTrading loop completed!")

# Check final state
print(f"\nFinal State:")
print(f"  - Iterations: {loop.state.iteration}")
print(f"  - Signals generated: {loop.metrics.total_signals}")
print(f"  - Orders created: {loop.metrics.total_orders}")
print(f"  - Fills received: {loop.metrics.total_fills}")
print(f"  - Daily P&L: ${loop.metrics.daily_pnl:.2f}")

Starting trading loop...


Trading loop completed!

Final State:
  - Iterations: 5
  - Signals generated: 2
  - Orders created: 2
  - Fills received: 2
  - Daily P&L: $-454.44


In [6]:
# Check portfolio state
portfolio = loop.portfolio
print(f"Portfolio Summary:")
print(f"  - Open positions: {len(portfolio.positions)}")
print(f"  - Closed positions: {len(portfolio.closed_positions)}")
print(f"  - Daily P&L: ${portfolio.daily_pnl:.2f}")
print(f"  - Total unrealized P&L: ${portfolio.total_unrealized_pnl:.2f}")

# Portfolio Greeks
greeks = portfolio.get_greeks()
print(f"\nPortfolio Greeks:")
print(f"  - Delta: {greeks.delta:.2f}")
print(f"  - Gamma: {greeks.gamma:.4f}")
print(f"  - Vega: {greeks.vega:.2f}")
print(f"  - Theta: {greeks.theta:.2f}")

Portfolio Summary:
  - Open positions: 0
  - Closed positions: 2
  - Daily P&L: $-454.44
  - Total unrealized P&L: $0.00

Portfolio Greeks:
  - Delta: 0.00
  - Gamma: 0.0000
  - Vega: 0.00
  - Theta: 0.00


## 4. State Management (M28)

Demonstrate state persistence and crash recovery.

In [7]:
# Check saved snapshots
state_manager = loop.state_manager

snapshot_count = state_manager.get_snapshot_count()
print(f"State snapshots saved: {snapshot_count}")

# List snapshot files
snapshot_files = sorted(Path(state_dir).glob("state_*.json"))
print(f"\nSnapshot files:")
for f in snapshot_files:
    print(f"  - {f.name}")

# Check order/fill history
print(f"\nOrder history: {len(state_manager.order_history)} orders")
print(f"Fill history: {len(state_manager.fill_history)} fills")
print(f"Last saved iteration: {state_manager.last_iteration}")

State snapshots saved: 5

Snapshot files:
  - state_20260126_042738_926378.json
  - state_20260126_042738_929522.json
  - state_20260126_042738_931144.json
  - state_20260126_042738_931891.json
  - state_20260126_042738_932577.json

Order history: 2 orders
Fill history: 2 fills
Last saved iteration: 5


In [8]:
# Load a snapshot to inspect
import json

latest_snapshot = state_manager.get_latest_snapshot_path()
print(f"Latest snapshot: {latest_snapshot.name}")

with open(latest_snapshot) as f:
    snapshot_data = json.load(f)

print(f"\nSnapshot contents:")
print(f"  - Iteration: {snapshot_data['iteration']}")
print(f"  - Timestamp: {snapshot_data['timestamp']}")
print(f"  - Daily P&L: ${snapshot_data['daily_pnl']:.2f}")
print(f"  - Positions: {len(snapshot_data['portfolio']['positions'])}")
print(f"  - Orders: {len(snapshot_data['order_history'])}")
print(f"  - Fills: {len(snapshot_data['fill_history'])}")

Latest snapshot: state_20260126_042738_932577.json

Snapshot contents:
  - Iteration: 5
  - Timestamp: 2026-01-26T04:27:38.932577+00:00
  - Daily P&L: $-454.44
  - Positions: 0
  - Orders: 2
  - Fills: 2


In [9]:
# Simulate crash recovery
print("=" * 50)
print("SIMULATING CRASH RECOVERY")
print("=" * 50)

# Record final state before "crash"
pre_crash_iteration = loop.state.iteration
pre_crash_positions = len(loop.portfolio.positions)
print(f"\nState before crash:")
print(f"  - Iteration: {pre_crash_iteration}")
print(f"  - Positions: {pre_crash_positions}")

# "Crash" - delete loop object
del loop
print("\n[CRASH - loop object deleted]\n")

# Create new trading loop - should recover state
paper_config_recovery = PaperConfig(
    state_dir=str(state_dir),
    save_state_interval=1,
    loop_interval_seconds=0,
    max_loop_iterations=pre_crash_iteration + 3,  # Run 3 more iterations
)

loop_recovered = TradingLoop(
    paper_config=paper_config_recovery,
    execution_config=execution_config,
    risk_config=risk_config,
    surface_provider=MockSurfaceProvider(),
    signal_generator=MockSignalGenerator(max_signals=0),  # No new signals
    order_router=PaperOrderRouter(execution_config),
)

print(f"State after recovery:")
print(f"  - Iteration: {loop_recovered.state.iteration}")
print(f"  - Positions: {len(loop_recovered.portfolio.positions)}")

# Verify recovery
assert loop_recovered.state.iteration == pre_crash_iteration, "Iteration not recovered!"
assert len(loop_recovered.portfolio.positions) == pre_crash_positions, "Positions not recovered!"
print("\n[SUCCESS] State recovered correctly!")

SIMULATING CRASH RECOVERY

State before crash:
  - Iteration: 5
  - Positions: 0

[CRASH - loop object deleted]

State after recovery:
  - Iteration: 5
  - Positions: 0

[SUCCESS] State recovered correctly!


In [10]:
# Continue trading after recovery
print("Continuing trading after recovery...\n")
loop_recovered.start()

print(f"\nFinal state after recovery:")
print(f"  - Iterations: {loop_recovered.state.iteration}")
print(f"  - Total snapshots: {loop_recovered.state_manager.get_snapshot_count()}")

Continuing trading after recovery...


Final state after recovery:
  - Iterations: 8
  - Total snapshots: 13


## 5. Position Tracking (M28)

Demonstrate mark-to-market position updates.

In [11]:
# Create a portfolio with a position
legs = [
    OptionLeg(
        symbol="SPY240315C00450000",
        qty=1,
        entry_price=5.00,
        strike=450.0,
        expiry=date(2024, 3, 15),
        right=OptionRight.CALL,
        greeks=Greeks(delta=0.50, gamma=0.02, vega=0.30, theta=-0.05),
    ),
    OptionLeg(
        symbol="SPY240315C00455000",
        qty=-1,
        entry_price=3.00,
        strike=455.0,
        expiry=date(2024, 3, 15),
        right=OptionRight.CALL,
        greeks=Greeks(delta=0.35, gamma=0.018, vega=0.25, theta=-0.04),
    ),
]

portfolio = Portfolio(daily_pnl=0.0)
portfolio = portfolio.add_position(
    legs=legs,
    position_id="vertical_spread_001",
    structure_type="VerticalSpread",
    max_loss=200.0,
)

print(f"Created portfolio with {len(portfolio.positions)} position(s)")
print(f"Position: {portfolio.positions[0].position_id}")
print(f"Structure: {portfolio.positions[0].structure_type}")

Created portfolio with 1 position(s)
Position: vertical_spread_001
Structure: VerticalSpread


In [12]:
# Create position tracker
tracker = PositionTracker(portfolio)

print(f"Position tracker initialized")
print(f"Portfolio positions: {len(tracker.portfolio.positions)}")

Position tracker initialized
Portfolio positions: 1


In [13]:
# Update with current surface (prices moved up)
surface_provider = MockSurfaceProvider()
surface = surface_provider.get_latest_surface()

print("Current surface prices:")
for _, row in surface.iterrows():
    print(f"  {row['option_symbol']}: bid={row['bid']:.2f}, ask={row['ask']:.2f}")

# Update positions
updated_portfolio = tracker.update_positions(surface)

print(f"\nPosition updated!")

Current surface prices:
  SPY240315C00450000: bid=5.05, ask=5.15
  SPY240315C00455000: bid=3.03, ask=3.11
  SPY240315P00445000: bid=2.78, ask=2.86

Position updated!


In [14]:
# Check position snapshot
snapshot = tracker.get_snapshot("vertical_spread_001")

print(f"Position Snapshot:")
print(f"  - Position ID: {snapshot.position_id}")
print(f"  - Entry value: ${snapshot.entry_value:.2f}")
print(f"  - Current value: ${snapshot.current_value:.2f}")
print(f"  - Unrealized P&L: ${snapshot.unrealized_pnl:.2f}")

print(f"\nMark prices:")
for symbol, price in snapshot.mark_prices.items():
    print(f"  {symbol}: ${price:.2f}")

print(f"\nCurrent Greeks:")
print(f"  - Delta: {snapshot.current_greeks.delta:.2f}")
print(f"  - Gamma: {snapshot.current_greeks.gamma:.4f}")
print(f"  - Vega: {snapshot.current_greeks.vega:.2f}")
print(f"  - Theta: {snapshot.current_greeks.theta:.2f}")

Position Snapshot:
  - Position ID: vertical_spread_001
  - Entry value: $200.00
  - Current value: $194.00
  - Unrealized P&L: $-6.00

Mark prices:
  SPY240315C00450000: $5.05
  SPY240315C00455000: $3.11

Current Greeks:
  - Delta: 15.00
  - Gamma: 0.2000
  - Vega: 5.00
  - Theta: -1.00


In [15]:
# Simulate multiple updates (prices evolving)
print("Simulating price evolution over 5 updates:\n")
print(f"{'Update':<8} {'Entry':>12} {'Current':>12} {'P&L':>12} {'Delta':>10}")
print("-" * 56)

for i in range(5):
    surface = surface_provider.get_latest_surface()
    tracker.update_positions(surface)
    snapshot = tracker.get_snapshot("vertical_spread_001")
    
    print(f"{i+1:<8} ${snapshot.entry_value:>10.2f} ${snapshot.current_value:>10.2f} "
          f"${snapshot.unrealized_pnl:>10.2f} {snapshot.current_greeks.delta:>10.2f}")

print(f"\nFinal unrealized P&L: ${tracker.get_total_unrealized_pnl():.2f}")

Simulating price evolution over 5 updates:

Update          Entry      Current          P&L      Delta
--------------------------------------------------------
1        $    200.00 $    196.00 $     -4.00      15.00
2        $    200.00 $    198.00 $     -2.00      15.00
3        $    200.00 $    200.00 $      0.00      15.00
4        $    200.00 $    202.00 $      2.00      15.00
5        $    200.00 $    204.00 $      4.00      15.00

Final unrealized P&L: $4.00


## 6. Trading Monitor (M28)

Demonstrate real-time monitoring and alerts.

In [16]:
# Track alerts
alerts_received = []
def alert_callback(alert):
    alerts_received.append(alert)

# Create monitor with alert thresholds
monitor = TradingMonitor(
    log_to_stdout=False,  # Suppress stdout for cleaner notebook output
    log_to_file=False,
    delta_threshold=50.0,   # Alert if |delta| > 50
    pnl_threshold=-100.0,   # Alert if P&L < -$100
    alert_callback=alert_callback,
)

print("Monitor created with thresholds:")
print(f"  - Delta threshold: 50.0")
print(f"  - P&L threshold: -$100.00")

Monitor created with thresholds:
  - Delta threshold: 50.0
  - P&L threshold: -$100.00


In [17]:
# Log some metrics
portfolio_small_delta = Portfolio(daily_pnl=50.0)
portfolio_small_delta = portfolio_small_delta.add_position(
    legs=[OptionLeg(
        symbol="SPY240315C00450000",
        qty=1,
        entry_price=5.00,
        strike=450.0,
        expiry=date(2024, 3, 15),
        right=OptionRight.CALL,
        greeks=Greeks(delta=0.30, gamma=0.02, vega=0.30, theta=-0.05),  # Small delta
    )],
    position_id="pos_001",
    structure_type="SingleLeg",
)

monitor.log_metrics(
    portfolio=portfolio_small_delta,
    iteration=1,
    signals_generated=2,
    orders_created=1,
    fills_received=1,
)

print(f"Logged iteration 1 - small delta position")
print(f"Alerts triggered: {len(alerts_received)}")

Logged iteration 1 - small delta position
Alerts triggered: 0


In [18]:
# Log metrics that trigger delta alert
portfolio_large_delta = Portfolio(daily_pnl=50.0)
portfolio_large_delta = portfolio_large_delta.add_position(
    legs=[OptionLeg(
        symbol="SPY240315C00450000",
        qty=2,
        entry_price=5.00,
        strike=450.0,
        expiry=date(2024, 3, 15),
        right=OptionRight.CALL,
        greeks=Greeks(delta=0.50, gamma=0.02, vega=0.30, theta=-0.05),  # 2 * 0.50 * 100 = 100 delta
    )],
    position_id="pos_002",
    structure_type="SingleLeg",
)

monitor.log_metrics(
    portfolio=portfolio_large_delta,
    iteration=2,
    signals_generated=1,
    orders_created=1,
    fills_received=1,
)

print(f"Logged iteration 2 - large delta position")
print(f"Portfolio delta: {portfolio_large_delta.get_greeks().delta:.2f}")
print(f"Alerts triggered: {len(alerts_received)}")

if alerts_received:
    print(f"\nLatest alert:")
    alert = alerts_received[-1]
    print(f"  - Level: {alert.level.name}")
    print(f"  - Message: {alert.message}")

ALERT [risk_monitor]: High delta exposure: 100.0 (threshold: ±50.0)


Logged iteration 2 - large delta position
Portfolio delta: 100.00
Alerts triggered: 1

Latest alert:
  - Level: WARNING
  - Message: High delta exposure: 100.0 (threshold: ±50.0)


In [19]:
# Log metrics that trigger P&L alert
portfolio_loss = Portfolio(daily_pnl=-150.0)  # Negative P&L

monitor.log_metrics(
    portfolio=portfolio_loss,
    iteration=3,
    signals_generated=0,
    orders_created=0,
    fills_received=0,
)

print(f"Logged iteration 3 - negative P&L")
print(f"Daily P&L: ${portfolio_loss.daily_pnl:.2f}")
print(f"Total alerts: {len(alerts_received)}")

print(f"\nAll alerts:")
for i, alert in enumerate(alerts_received, 1):
    print(f"  {i}. [{alert.level.name}] {alert.message}")

ALERT [risk_monitor]: Daily P&L below threshold: $-150.00 (threshold: $-100.00)


Logged iteration 3 - negative P&L
Daily P&L: $-150.00
Total alerts: 2

All alerts:
  1. [WARNING] High delta exposure: 100.0 (threshold: ±50.0)
  2. [WARNING] Daily P&L below threshold: $-150.00 (threshold: $-100.00)


In [20]:
# Get monitor summary
summary = monitor.get_summary()

print("Monitor Summary:")
print(f"  - Total iterations logged: {summary['total_iterations']}")
print(f"  - Total alerts: {summary['total_alerts']}")
print(f"  - Alerts by level:")
for level, count in summary['alerts_by_level'].items():
    print(f"      {level}: {count}")

Monitor Summary:
  - Total iterations logged: 3
  - Total alerts: 2
  - Alerts by level:
      info: 0
      error: 0
      critical: 0


In [21]:
# View metrics history
print("Metrics History:")
print(f"{'Iter':<6} {'Signals':>8} {'Orders':>8} {'Fills':>8} {'P&L':>10} {'Delta':>10}")
print("-" * 56)

for metrics in monitor.metrics_history:
    print(f"{metrics.iteration:<6} {metrics.signals_generated:>8} {metrics.orders_created:>8} "
          f"{metrics.fills_received:>8} ${metrics.daily_pnl:>8.2f} {metrics.net_delta:>10.2f}")

Metrics History:
Iter    Signals   Orders    Fills        P&L      Delta
--------------------------------------------------------
1             2        1        1 $   50.00      30.00
2             1        1        1 $   50.00     100.00
3             0        0        0 $ -150.00       0.00


## 7. Complete Integration Demo

Run a complete trading session with all M27-M28 features.

In [22]:
# Create fresh state directory
integration_dir = tempfile.mkdtemp()
integration_state_dir = Path(integration_dir) / "state"
integration_state_dir.mkdir()

# Config for integration demo
integration_config = PaperConfig(
    state_dir=str(integration_state_dir),
    save_state_interval=1,
    loop_interval_seconds=0,
    max_loop_iterations=3,
)

# Create custom monitor
demo_monitor = TradingMonitor(
    log_to_stdout=False,
    delta_threshold=100.0,
    pnl_threshold=-500.0,
)

# Create trading loop
demo_loop = TradingLoop(
    paper_config=integration_config,
    execution_config=execution_config,
    risk_config=risk_config,
    surface_provider=MockSurfaceProvider(),
    signal_generator=MockSignalGenerator(max_signals=2),
    order_router=PaperOrderRouter(execution_config),
    monitor=demo_monitor,
)

print("Integration demo setup complete")

Integration demo setup complete


In [23]:
# Run trading session
print("=" * 60)
print("PAPER TRADING SESSION")
print("=" * 60)

demo_loop.start()

print(f"\nSession Complete!")
print(f"\nMetrics:")
print(f"  - Iterations: {demo_loop.state.iteration}")
print(f"  - Signals: {demo_loop.metrics.total_signals}")
print(f"  - Orders: {demo_loop.metrics.total_orders}")
print(f"  - Fills: {demo_loop.metrics.total_fills}")
print(f"  - Daily P&L: ${demo_loop.metrics.daily_pnl:.2f}")

print(f"\nPortfolio:")
print(f"  - Positions: {len(demo_loop.portfolio.positions)}")
greeks = demo_loop.portfolio.get_greeks()
print(f"  - Delta: {greeks.delta:.2f}")
print(f"  - Vega: {greeks.vega:.2f}")

print(f"\nState Persistence:")
print(f"  - Snapshots: {demo_loop.state_manager.get_snapshot_count()}")
print(f"  - Orders recorded: {len(demo_loop.state_manager.order_history)}")
print(f"  - Fills recorded: {len(demo_loop.state_manager.fill_history)}")

print(f"\nMonitoring:")
print(f"  - Metrics logged: {len(demo_monitor.metrics_history)}")
print(f"  - Alerts: {len(demo_monitor.alert_history)}")

PAPER TRADING SESSION

Session Complete!

Metrics:
  - Iterations: 3
  - Signals: 2
  - Orders: 2
  - Fills: 2
  - Daily P&L: $-454.44

Portfolio:
  - Positions: 0
  - Delta: 0.00
  - Vega: 0.00

State Persistence:
  - Snapshots: 3
  - Orders recorded: 2
  - Fills recorded: 2

Monitoring:
  - Metrics logged: 3
  - Alerts: 0


In [24]:
# Final verification
print("\n" + "=" * 60)
print("VERIFICATION")
print("=" * 60)

checks = [
    ("Trading loop completed", demo_loop.state.iteration == 3),
    ("Signals were generated", demo_loop.metrics.total_signals > 0),
    ("State was persisted", demo_loop.state_manager.get_snapshot_count() >= 3),
    ("Metrics were logged", len(demo_monitor.metrics_history) == 3),
    ("Portfolio tracks positions", len(demo_loop.portfolio.positions) >= 0),
]

all_passed = True
for check_name, passed in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check_name}")
    if not passed:
        all_passed = False

print(f"\n{'All checks passed!' if all_passed else 'Some checks failed!'}")


VERIFICATION
  [PASS] Trading loop completed
  [PASS] Signals were generated
  [PASS] State was persisted
  [PASS] Metrics were logged
  [PASS] Portfolio tracks positions

All checks passed!


## Summary

This notebook demonstrated:

**M27 - Paper Trading Loop:**
- `TradingLoop` orchestrates the full trading cycle
- `PaperOrderRouter` simulates order execution with slippage/fees
- Signal generation → order creation → risk check → fill → position update
- Kill switch integration for daily loss limits

**M28 - State Management & Monitoring:**
- `StateManager` persists state to JSON snapshots
- Crash recovery restores iteration, portfolio, orders, and fills
- `PositionTracker` provides mark-to-market valuation
- `TradingMonitor` logs metrics and sends threshold alerts

The paper trading infrastructure is now complete and ready for use!